# PDAC multi-omic pipeline in Colab

Runs `pdac_pipeline.py` with results saved to Google Drive, so a disconnected
session picks up where it stopped. Put these two files in one Drive folder
first:

- `pdac_pipeline.py`
- `43018_2023_697_MOESM3_ESM.xlsx`

Cost: one leave-one-out seed is about 9 CPU-hours, so roughly 4–5 hours on a
free 2-core runtime, or about 1 hour on a Colab Pro high-RAM runtime with 8
cores. Everything is CPU-only; do not pick a GPU runtime.

In [ ]:
!pip -q install imbalanced-learn xgboost lightgbm openpyxl
import os; print('cores available:', os.cpu_count())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

FOLDER = '/content/drive/MyDrive/pdac'            # <- the Drive folder holding both files
DATA   = f'{FOLDER}/43018_2023_697_MOESM3_ESM.xlsx'
SCRIPT = f'{FOLDER}/pdac_pipeline.py'
NJOBS  = os.cpu_count()

# The manuscript's protocol is leave-one-out. For a quicker robustness run use the
# second pair of lines instead (separate folder: one folder per CV scheme).
OUT, CV = f'{FOLDER}/results_loo', ''
# OUT, CV = f'{FOLDER}/results_rskf', '--cv rskf --cv-folds 5 --cv-repeats 5'

assert os.path.exists(DATA) and os.path.exists(SCRIPT), 'put both files in FOLDER first'

## 1. Audit (seconds)
Read the column warnings and the label-version table before going further.
`published_matches` should say `moesm3` for every analyte: the published
Table 1 was computed on the released labels.

In [ ]:
!python "$SCRIPT" --data "$DATA" --outdir "$OUT" $CV --parts audit

## 2. Smoke test (a few minutes)
Small models on two small analytes. Confirms every method runs with the
installed packages. It writes to its own folder.

In [ ]:
!python "$SCRIPT" --data "$DATA" --outdir /content/smoke $CV --smoke --n-jobs $NJOBS

## 3. Full run, one analyte at a time
Each finished run is cached on Drive. If the session disconnects, run this cell
again; completed runs are skipped. Small analytes come first so you see results
early. `P3` (stacking) is the slowest method; leave it out with
`--methods P1 P2 P4 P5 P6` if you must, and say so in the paper.

In [ ]:
ANALYTES = ['RNA_Fusion', 'INDEL', 'Clinical', 'SNV', 'CNV', 'Pathology',
            'Plasma_Protein', 'Plasma_Lipid', 'Tissue_Protein', 'RNA', 'Multiomic']
PARTS = 'baseline pipelines ablation alpha_sweep'
SEEDS = '42'            # add more seeds later, e.g. '42 43 44 45 46', for seed_sensitivity

for a in ANALYTES:
    print('=' * 20, a, flush=True)
    !python -u "$SCRIPT" --data "$DATA" --outdir "$OUT" $CV --analytes $a --parts $PARTS --seeds $SEEDS --n-jobs $NJOBS

## 4. Tables and figures
Rebuilds everything in `report/` from the cached predictions. Works on a
partial run too.

In [ ]:
!python "$SCRIPT" --data "$DATA" --outdir "$OUT" $CV --parts report
import pandas as pd
pd.set_option('display.width', 200)
print('Reproduction of the released models vs the published Table 1')
display(pd.read_csv(f'{OUT}/report/table2_reproduction.csv').round(3))
print('Step 2 (released models, nested selection) vs Step 3 (fold-internal pipelines)')
display(pd.read_csv(f'{OUT}/report/table3_step2_vs_step3.csv').round(3))
print('Ablation')
display(pd.read_csv(f'{OUT}/report/table5_ablation.csv').round(3))

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob(f'{OUT}/report/*.png')):
    print(f); display(Image(f, width=900))

## 5. Optional: permutation null
Re-runs the methods on shuffled labels. Expensive (about 20 CPU-hours per
shuffle for all analytes). Start with one analyte and a few shuffles; the
cache lets you add more later.

In [ ]:
!python -u "$SCRIPT" --data "$DATA" --outdir "$OUT" $CV --analytes Multiomic --parts nulltest --null-perms 10 --n-jobs $NJOBS
pd.read_csv(f'{OUT}/report/null_check.csv').round(3)

## 6. Download the report folder
Send `report.zip` for the table and text rebuild.

In [ ]:
!cd "$OUT" && rm -f report.zip && zip -qr report.zip report && ls -lh report.zip
from google.colab import files
files.download(f'{OUT}/report.zip')